# Sprint 1: Tokenization Analysis

This notebook prepares the first descriptive analysis for the thesis project. The goal is not to extract embeddings yet, but to measure how three BERT-like tokenizers fragment the Turkish words in the AnlamVer semantic similarity dataset.

The analysis covers:

- loading and cleaning AnlamVer,
- converting comma-decimal numeric columns such as `Sim` and `Rel` to floats,
- extracting the unique Turkish words from `W1` and `W2`,
- tokenizing all word pairs with BERTurk, multilingual BERT, and XLM-RoBERTa,
- saving a processed pair-level dataset and thesis-ready summary tables.


## Methodological note

AnlamVer contains Turkish word pairs with human semantic similarity (`Sim`) and relatedness (`Rel`) judgments. Since Turkish is morphologically rich and agglutinative, words may be split into multiple subword tokens by transformer tokenizers. This notebook treats tokenization fragmentation as a descriptive property of the dataset and as preparation for later embedding-composition experiments.

The later embedding experiment will compare pooling strategies such as first-token, last-token, mean, and max pooling. This notebook deliberately stops before embedding extraction.

In [ ]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

def find_project_root(start: Path) -> Path:
    """Find the project root whether the notebook is run from root or notebooks/."""
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "thesis_project_context.md").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "anlamver_tokenization_analysis.csv"

pd.set_option("display.max_colwidth", 120)


## Load AnlamVer

The project may contain the dataset in either the new `data/raw/` layout or the older `datasets/` layout. The loader below checks both locations and uses the first available file. The original AnlamVer CSV uses semicolon separators and Turkish/Windows encoding, so the helper tries those settings after a general automatic read attempt.

In [ ]:
INPUT_CANDIDATES = [
    RAW_DIR / "anlamver.csv",
    RAW_DIR / "anlamver_with_tokens.csv",
    PROJECT_ROOT / "datasets" / "anlamver-final.csv",
    PROJECT_ROOT / "datasets" / "anlamver_with_tokens.csv",
]

def read_anlamver(path: Path) -> pd.DataFrame:
    """Read AnlamVer from the known project CSV formats."""
    try:
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=";", encoding="windows-1254")
    except pd.errors.ParserError:
        return pd.read_csv(path, sep=";", encoding="windows-1254")

input_path = next((path for path in INPUT_CANDIDATES if path.exists()), None)
if input_path is None:
    searched = "\n".join(str(path) for path in INPUT_CANDIDATES)
    raise FileNotFoundError(f"Could not find an AnlamVer input file. Searched:\n{searched}")

df = read_anlamver(input_path)
print(f"Loaded {len(df):,} rows from {input_path}")
df.head()


## Clean numeric columns

The AnlamVer ratings are stored with comma decimals, for example `0,166666667`. These need to be converted to numeric floats before later correlation analyses. Here the cleaning is limited to columns that are expected to be numeric and present in the dataset.

In [ ]:
def comma_decimal_to_float(series: pd.Series) -> pd.Series:
    """Convert strings such as '1,25' to floats while preserving missing values."""
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )

numeric_columns = [
    "Sim", "Rel", "AVG-C", "W1F", "W2F", "RWMin", "DGMax", "IGMax",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = comma_decimal_to_float(df[column])

df[[column for column in ["W1", "W2", "Sim", "Rel"] if column in df.columns]].head()


## Extract unique words

The tokenizers are applied to word types first, not directly to every pair occurrence. This avoids repeated work and makes it easy to summarize fragmentation at the unique-word level.

In [ ]:
required_columns = {"W1", "W2"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

unique_words = pd.Index(pd.concat([df["W1"], df["W2"]]).dropna().astype(str).str.strip().unique()).sort_values()

print(f"Word pairs: {len(df):,}")
print(f"Word occurrences: {len(df) * 2:,}")
print(f"Unique words: {len(unique_words):,}")
unique_words[:10].tolist()


## Tokenizers

Three tokenizers are compared:

- **BERTurk**: `dbmdz/bert-base-turkish-cased`, a Turkish-specific BERT model.
- **mBERT**: `bert-base-multilingual-cased`, a multilingual WordPiece model.
- **XLM-RoBERTa**: `xlm-roberta-base`, a multilingual SentencePiece model.

For each tokenizer, a word is considered *split* when it receives more than one subtoken. A pair is considered split when either `W1` or `W2` is split.

In [ ]:
MODELS = {
    "berturk": "dbmdz/bert-base-turkish-cased",
    "mbert": "bert-base-multilingual-cased",
    "xlmr": "xlm-roberta-base",
}

def tokenize_words(model_name: str, words: pd.Index) -> pd.DataFrame:
    """Return one row per unique word with tokenizer tokens and token counts."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rows = []
    for word in words:
        tokens = tokenizer.tokenize(word)
        rows.append({
            "word": word,
            "tokens": tokens,
            "subtoken_count": len(tokens),
            "is_split": len(tokens) > 1,
        })
    return pd.DataFrame(rows)

word_tokenizations = {
    short_name: tokenize_words(model_name, unique_words)
    for short_name, model_name in MODELS.items()
}

word_tokenizations["berturk"].head()


## Add tokenizer features to word pairs

The processed pair-level dataset keeps the original AnlamVer columns and adds model-specific tokenization features. Token lists are stored as space-separated strings in the CSV so they remain easy to inspect in spreadsheet software.

In [ ]:
analysis_df = df.copy()

for short_name, token_df in word_tokenizations.items():
    token_lookup = token_df.set_index("word")

    for side, source_column in [("word1", "W1"), ("word2", "W2")]:
        words = analysis_df[source_column].astype(str).str.strip()
        analysis_df[f"{short_name}_{side}_tokens"] = words.map(token_lookup["tokens"]).apply(" ".join)
        analysis_df[f"{short_name}_{side}_subtoken_count"] = words.map(token_lookup["subtoken_count"]).astype("Int64")

    w1_count = analysis_df[f"{short_name}_word1_subtoken_count"]
    w2_count = analysis_df[f"{short_name}_word2_subtoken_count"]
    analysis_df[f"{short_name}_pair_total_subtoken_count"] = w1_count + w2_count
    analysis_df[f"{short_name}_pair_max_subtoken_count"] = pd.concat([w1_count, w2_count], axis=1).max(axis=1)
    analysis_df[f"{short_name}_pair_clean"] = (w1_count == 1) & (w2_count == 1)
    analysis_df[f"{short_name}_pair_split"] = ~analysis_df[f"{short_name}_pair_clean"]

analysis_columns = [
    "W1", "W2", "Sim", "Rel",
    "berturk_word1_tokens", "berturk_word2_tokens",
    "berturk_word1_subtoken_count", "berturk_word2_subtoken_count",
    "berturk_pair_total_subtoken_count", "berturk_pair_max_subtoken_count",
    "berturk_pair_clean", "berturk_pair_split",
]
analysis_df[[column for column in analysis_columns if column in analysis_df.columns]].head()


## Summary tables

The tables below are designed for the Methods and Data Exploration sections. They describe how often words and pairs are split, how severe the fragmentation is, and which words are most fragmented for each tokenizer.

In [ ]:
def subtoken_distribution(tokenizations: dict[str, pd.DataFrame], pair_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for short_name, token_df in tokenizations.items():
        unique_counts = token_df["subtoken_count"].value_counts().sort_index()
        occurrence_counts = pd.concat([
            pair_df[f"{short_name}_word1_subtoken_count"],
            pair_df[f"{short_name}_word2_subtoken_count"],
        ]).value_counts().sort_index()

        all_counts = sorted(set(unique_counts.index).union(set(occurrence_counts.index)))
        for subtoken_count in all_counts:
            unique_word_count = int(unique_counts.get(subtoken_count, 0))
            word_occurrence_count = int(occurrence_counts.get(subtoken_count, 0))
            rows.append({
                "model": short_name,
                "subtoken_count": int(subtoken_count),
                "unique_words": unique_word_count,
                "unique_word_percentage": round(unique_word_count / len(token_df) * 100, 2),
                "word_occurrences": word_occurrence_count,
                "word_occurrence_percentage": round(word_occurrence_count / (len(pair_df) * 2) * 100, 2),
            })
    return pd.DataFrame(rows)

def split_pair_percentages(pair_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for short_name in MODELS:
        split_pairs = pair_df[f"{short_name}_pair_split"].sum()
        clean_pairs = pair_df[f"{short_name}_pair_clean"].sum()
        rows.append({
            "model": short_name,
            "word_pairs": len(pair_df),
            "clean_pairs": int(clean_pairs),
            "split_pairs": int(split_pairs),
            "split_pair_percentage": round(split_pairs / len(pair_df) * 100, 2),
        })
    return pd.DataFrame(rows)

def three_plus_token_words(tokenizations: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for short_name, token_df in tokenizations.items():
        n_words = (token_df["subtoken_count"] >= 3).sum()
        rows.append({
            "model": short_name,
            "unique_words": len(token_df),
            "three_plus_token_words": int(n_words),
            "percentage": round(n_words / len(token_df) * 100, 2),
        })
    return pd.DataFrame(rows)

def most_fragmented_words(tokenizations: dict[str, pd.DataFrame], top_n: int = 25) -> pd.DataFrame:
    frames = []
    for short_name, token_df in tokenizations.items():
        top_words = token_df.sort_values(["subtoken_count", "word"], ascending=[False, True]).head(top_n).copy()
        top_words.insert(0, "model", short_name)
        top_words["tokens"] = top_words["tokens"].apply(" ".join)
        frames.append(top_words)
    return pd.concat(frames, ignore_index=True)

summary_distribution = subtoken_distribution(word_tokenizations, analysis_df)
summary_split_pairs = split_pair_percentages(analysis_df)
summary_three_plus = three_plus_token_words(word_tokenizations)
summary_most_fragmented = most_fragmented_words(word_tokenizations)


In [ ]:
summary_distribution


In [ ]:
summary_split_pairs


In [ ]:
summary_three_plus


In [ ]:
summary_most_fragmented.head(30)


## Save processed data and tables

The processed dataset is saved at pair level. Summary tables are saved separately under `outputs/tables/` so they can be reused in the thesis text, slides, or later analysis notebooks.

In [ ]:
analysis_df.to_csv(PROCESSED_PATH, index=False)

summary_distribution.to_csv(TABLE_DIR / "tokenization_subtoken_count_distribution.csv", index=False)
summary_split_pairs.to_csv(TABLE_DIR / "tokenization_split_pair_percentage.csv", index=False)
summary_three_plus.to_csv(TABLE_DIR / "tokenization_three_plus_token_words.csv", index=False)
summary_most_fragmented.to_csv(TABLE_DIR / "tokenization_most_fragmented_words.csv", index=False)

print(f"Saved processed dataset to: {PROCESSED_PATH}")
print(f"Saved summary tables to: {TABLE_DIR}")


## Interpretation checklist

Use these points when writing up the Sprint 1 results:

- A high split-pair percentage means many AnlamVer pairs require a subtoken-to-word composition decision before embeddings can be compared.
- The `3+` token subset identifies the words where first-token or last-token pooling is most likely to discard information.
- Differences between BERTurk, mBERT, and XLM-R show whether Turkish-specific tokenization reduces fragmentation compared with multilingual tokenization.
- These statistics are descriptive. Fragmentation should not be interpreted as causing lower human similarity ratings without later modeling and controls.